In [1]:
import gaiadr3_analysis.gaia_input as gi
import gaiadr3_analysis.mean_photometry as mphot
import pandas as pd
import numpy as np

Maintenance with possible short-time disconnections: 29 June 2026 18:00–20:00 CEST


In [3]:
#Parse to pandas dataframe
#sep = '\s+' sets the seperator/delimiter for the data to be one or multiple spaces
df = pd.read_csv("stars.dat", sep = "\s+", names=['name_id', 'name_val', 'ra_h', 'ra_m', 'ra_s', 'dec_h', 'dec_m', 'dec_s', 'source', 'full_name'])
df['full_name'] = (df['name_id'] +" "+ df['name_val'])
print(df.head())

  name_id name_val  ra_h  ra_m   ra_s  dec_h  dec_m  dec_s       source  \
0   BD+60      499     2    32  16.75     61     33   15.0  tablec1.dat   
1   BD-13     4930    18    18  52.67    -13     49   42.6  tablec1.dat   
2  CPD-28     2561     7    55  52.85    -28     37   46.8  tablec1.dat   
3      HD      108     0     6   3.39     63     40   46.8  tablec1.dat   
4      HD    13745     2    15  45.93     55     59   46.7  tablec1.dat   

     full_name  
0    BD+60 499  
1   BD-13 4930  
2  CPD-28 2561  
3       HD 108  
4     HD 13745  


<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
C:\Users\matth\AppData\Local\Temp\ipykernel_31360\3449405957.py:3: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv("stars.dat", sep = "\s+", names=['name_id', 'name_val', 'ra_h', 'ra_m', 'ra_s', 'dec_h', 'dec_m', 'dec_s', 'source', 'full_name'])


In [4]:
#Compare HD to Gaia
indentifier_error = []
no_cross_match = []
for sim_id in df['full_name']:
    gaia_id = gi.resolve_id(sim_id)
    #print(f"SIMBAD ID: {sim_id}, Gaia DR3 ID: {gaia_id}")
    if gaia_id == None:
        no_cross_match.append(sim_id)
    elif gaia_id == -1:
        indentifier_error.append(sim_id)
    

DALServiceError: Unable to access the capabilities endpoint at:
- https://simbad.cds.unistra.fr/simbad/sim-tap/capabilities: HTTP Code: 503 Service Unavailable Response body: <!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">
<html><head>
<title>503 Service Unavailable</title>
</head><body>
<h1>Service Unavailable</h1>
<p>The server is temporarily unable to service your
request due to maintenance downtime or capacity
problems. Please try again later.</p>
<hr>
<address>Apache/2.4.52 (Ubuntu) Server at simbad.cds.unistra.fr Port 443</address>
</body></html>


This could mean:
1. The service URL is incorrect
2. The service is temporarily unavailable
3. The service doesn't support this protocol
4. If a 503 was encountered, retry after the suggested delay.


In [5]:
print(indentifier_error)

[]


In [6]:
print(no_cross_match)
#Using SIMBAD I checked and all of these do not have Gaia IDs on the site

[]


In [ ]:
#Cross match with RA & DEC
ra_dec = []
for name in no_cross_match:
    temp = df[df['full_name'] == name]
    print(temp)
    print(temp['ra_h'].iloc[0], )

In [7]:
from astropy.coordinates import SkyCoord  # High-level coordinates
from astropy.coordinates import ICRS, Galactic, FK4, FK5  # Low-level frames
from astropy.coordinates import Angle, Latitude, Longitude  # Anglesstars = gi.get_dataframe()
import astropy.units as u

HD218195_coord = SkyCoord('23 5 12.93', '58 14 29.3', unit=(u.hourangle, u.deg))
HD17505_coord = SkyCoord('2 51 7.98', '60 25 3.9', unit=(u.hourangle, u.deg))
HD191201_coord = SkyCoord('20 7 23.69', '35 43 5.9', unit=(u.hourangle, u.deg))
HD193322_coord = SkyCoord('20 18 6.99', '40 43 55.5', unit=(u.hourangle, u.deg))

HD218195_df = gi.find_star(coordinates = HD218195_coord, save_file = True, file_name="HD218195")
HD17505_df = gi.find_star(coordinates = HD17505_coord, save_file = True, file_name="HD17505")
HD191201_df = gi.find_star(coordinates = HD191201_coord, save_file = True, file_name="HD191201")
HD193322_df = gi.find_star(coordinates = HD193322_coord, save_file = True, file_name="HD193322")

In [ ]:
query = f"""
SELECT TOP 10 source_id, ra, dec, pmra, pmdec,
       parallax, parallax_error,
       phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag,
       ruwe, radial_velocity, phot_variable_flag
FROM gaiadr3.gaia_source
WHERE CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', 346.3038749999999, 58.24147222222222, 0.0001)
) = 1
OR CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', 42.783249999999995, 60.41775, 0.0001)
) = 1
OR CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', 301.8487083333333, 35.71830555555556, 0.0001)
) = 1
OR CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', 304.52912499999996, 40.732083333333335, 0.0001)
) = 1
"""
"""
print(HD218195_coord.ra.deg)
print(HD218195_coord.dec.deg)
print(HD17505_coord.ra.deg)
print(HD17505_coord.dec.deg)
print(HD191201_coord.ra.deg)
print(HD191201_coord.dec.deg)
print(HD193322_coord.ra.deg)
print(HD193322_coord.dec.deg)
"""
crossmatch_df = gi.query_by_adql(query, save_file = True, file_name = "VizieRSurvey_SIMBAD/crossmatchRaDec")

In [ ]:
print(crossmatch_df.head())
#Cannot turn spectroscopic binaries into Gaia IDs since they represent more than one Gaia ID. e.g. A and B